# Проверка CSV до обучения
Часть 1 по слайду. По умолчанию проверяются 12 000 СИНТЕТИЧЕСКИХ заявок, созданных генератором проекта. Каждая модификация данных фиксируется явно. Строки, включая критические, при проверке сохраняются.

Владелец артефакта Климова Дарья. Независимую проверку выполняет Савенкова Юлия. При обсуждении требований участвует Ким Эвелина.

In [ ]:
from pathlib import Path
from uuid import uuid4
from IPython.display import display
from triage import read_csv, is_synthetic
from create_synthetic_csv import DEPARTMENTS
from data_checks import inspect_data

DATA_CSV = Path("data/synthetic/tickets_synthetic.csv")
frame = read_csv(DATA_CSV)
demo = is_synthetic(frame)
mode = "SYNTHETIC" if demo else "historical"
# Для реального CSV укажите согласованный список подразделений.
EXPECTED_DEPARTMENTS = list(DEPARTMENTS) if demo else None
report = inspect_data(frame, departments=EXPECTED_DEPARTMENTS, min_chars=20)
print(mode, "· shape", frame.shape)
display(frame.head())
display(report["fields"])

## Уникальность и дефекты
duplicate_row считается по пяти основным полям после удаления крайних пробелов и нормализации приоритета. duplicate_text сравнивает объединённый текст без учёта регистра и повторных пробелов. Значения в отчёте показывают число участвующих строк. short_input отмечает от 1 до 19 символов. Это диагностический порог, он не является правилом удаления.

In [ ]:
print("Уникальных ID:", frame.id.nunique(), "из", len(frame))
display(report["counts"].rename("Количество строк").to_frame())
print("Словарь подразделений проверен:", report["department_dictionary_checked"])
display(report["issues"].head(20))

## Распределения и длины
Для синтетического CSV классы и частоты заданы генератором. Эти числа описывают учебный набор. После появления исторических данных распределения нужно измерить заново. Для исторического CSV первичный просмотр схемы не используется для подбора модели. Содержательные ML-гипотезы проверяются на train и validation.

In [ ]:
display(report["priority_counts"].rename("Количество").to_frame())
display(report["department_counts"].rename("Количество").to_frame())
display(report["text_lengths"].rename("Символов в теме и тексте").to_frame())

## Сохранение проверки и следующий шаг
Проверка выполняется без изменения frame. Пустые поля и короткие тексты разбираются отдельно. Критические заявки сохраняются. В experiment.ipynb применяются только описанные преобразования и проверки. Все ошибки из этого notebook требуют решения до обучения.

Для исходного синтетического CSV результат первого выполнения также сохранён в docs/data_audit.md. Локальный отчёт этого прогона создаётся ниже. Следующий шаг Климовой Дарьи состоит в запуске train/validation/test. Проверка на исторических обращениях ожидает фактических данных.

In [ ]:
out = Path("runs") / mode / ("eda-" + uuid4().hex)
out.mkdir(parents=True, exist_ok=False)
report["fields"].to_csv(out / "fields.csv")
report["counts"].to_csv(out / "checks.csv")
report["issues"].to_csv(out / "issues.csv", index=False)
report["priority_counts"].to_csv(out / "priority_counts.csv")
report["department_counts"].to_csv(out / "department_counts.csv")
print(mode, "· отчёты сохранены в", out)